In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
SILVER_PATH_ATIVO_PASSIVO   = "workspace.case_spark_cvm.silver_cvm_fii_ativo_passivo"
SILVER_PATH_COMPLEMENTO     = "workspace.case_spark_cvm.silver_cvm_fii_complemento"
SILVER_PATH_INDICADORES     = "workspace.case_spark_cvm.silver_dados_indicadores_economicos"


NOME_TABELA  = f"gold_cubo_fii_mensal" 
GOLD_PATH    = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

In [0]:
df_ativo_passivo_silver = spark.read.table(SILVER_PATH_ATIVO_PASSIVO)
df_complemento_silver   = spark.read.table(SILVER_PATH_COMPLEMENTO)
df_indicadores_silver   = spark.read.table(SILVER_PATH_INDICADORES)

In [0]:
df_indicadores_silver.display()

#### 3.1 Criando CDI Acumulado Mes

In [0]:
df_indicadores = df_indicadores_silver\
    .withColumn("ano_mes", f.date_trunc("month", f.col("data")).cast(t.DateType()))\
    .withColumn(
        "fator_cdi_dia",
        1 + f.coalesce(f.col("valor_cdi"), f.lit(0)) / 100
    )\
    .groupby("ano_mes")\
    .agg(
        (f.exp(f.sum(f.log(f.col("fator_cdi_dia")))) - 1).alias("cdi_acum_mes"),
        f.last("ipca_mensal", ignorenulls=True).alias("ipca_mensal"),
        f.last("ipca_anual", ignorenulls=True).alias("ipca_anual")
    )

#### 3.2 Criando CDI Acumulado 12 Meses

In [0]:
window_12m = Window.orderBy("ano_mes").rowsBetween(-11, 0)

df_indicadores = df_indicadores.withColumn(
    "cdi_acum_12m",
    (f.exp(f.sum(f.log(1 + f.col("cdi_acum_mes"))).over(window_12m)) -1 )
)

### 4. Join

In [0]:
gold_cubo_fii_mensal = df_complemento_silver.alias("c") \
    .join(
        df_ativo_passivo_silver.alias("a"),
        (f.col("a.cnpj_fundo_classe") == f.col("c.cnpj_fundo_classe")) &
        (f.col("a.data_referencia") == f.col("c.data_referencia")),
        "left"
    ) \
    .join(
        df_indicadores.alias("i"),
        f.date_trunc("month", f.col("c.data_referencia")).cast(t.DateType()) == f.col("i.ano_mes"),
        "left"
    )\
    .drop(f.col("a.cnpj_fundo_classe"), f.col("a.data_referencia"))

In [0]:
gold_cubo_fii_mensal.display()

### 5. Rentabilidade e Dividend Yield

In [0]:
gold_cubo_fii_mensal = gold_cubo_fii_mensal\
  .withColumn(
    "rentabilidade_efetiva_mes",
    f.col("percentual_rentabilidade_efetiva_mes")
  )\
  .withColumn(
    "rentabilidade_patrimonial_mes",
    f.col("percentual_rentabilidade_patrimonial_mes")
  )\
  .withColumn(
    "dividend_yield_mes",
    f.col("percentual_dividend_yield_mes")
  )

In [0]:
window_fundo_12m = Window.partitionBy("cnpj_fundo_classe").orderBy("data_referencia").rowsBetween(-11,0)

gold_cubo_fii_mensal = gold_cubo_fii_mensal\
  .withColumn(
    "rentabilidade_efetiva_12m",
    f.when(
      f.col("rentabilidade_efetiva_mes").isNotNull(),
      f.exp(
        f.sum(
          f.log(f.when(f.col("rentabilidade_efetiva_mes") > -1, 1 + f.col("rentabilidade_efetiva_mes")))
        ).over(window_fundo_12m)
      ) -1 
    )
  )\
  .withColumn(
    "dividend_yield_12m",
    f.sum("dividend_yield_mes").over(window_fundo_12m)
  )\
  .withColumn(
    "alpha_vs_cdi_mes",
    f.col("rentabilidade_efetiva_mes") - f.col("cdi_acum_mes")
  )\
  .withColumn(
    "alpha_vs_cdi_12m",
    f.col("rentabilidade_efetiva_12m") - f.col("cdi_acum_12m")
  )

### 6. Patrimônio e Cotas

In [0]:
gold_cubo_fii_mensal = gold_cubo_fii_mensal\
    .withColumnRenamed("percentual_despesas_taxa_administracao", "taxa_administracao_pct") \
    .withColumnRenamed("percentual_amortizacao_cotas_mes", "amortizacao_cotas_mes")

In [0]:
window_fundo = Window.partitionBy("cnpj_fundo_classe").orderBy("data_referencia")

gold_cubo_fii_mensal = gold_cubo_fii_mensal\
    .withColumn("pl_lag", f.lag("patrimonio_liquido").over(window_fundo))\
    .withColumn("vpc_lag", f.lag("valor_patrimonial_cotas").over(window_fundo))\
    .withColumn(
        "variacao_pl_mes",
        f.when(f.col("pl_lag").isNotNull(), f.try_divide(f.col("patrimonio_liquido"), f.col("pl_lag")) - 1)
    )\
    .withColumn(
        "variacao_vpc_mes",
        f.when(f.col("vpc_lag").isNotNull(), f.try_divide(f.col("valor_patrimonial_cotas"), f.col("vpc_lag")) - 1)
    )

### 7. Cotistas

In [0]:

window_fundo = Window.partitionBy("cnpj_fundo_classe").orderBy("data_referencia")

window_fill = Window.partitionBy("cnpj_fundo_classe").orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, 0)

gold_cubo_fii_mensal = gold_cubo_fii_mensal\
  .withColumn(
    "total_cotistas", 
    f.last("total_numero_cotistas", ignorenulls=True).over(window_fill)
    )\
  .withColumn(
    "cotistas_pessoa_fisica", 
    f.last("numero_cotistas_pessoa_fisica", ignorenulls=True).over(window_fill)
    )\
  .withColumn(
    "cotistas_pessoa_juridica", 
    f.last("numero_cotistas_pessoa_juridica_nao_financeira", ignorenulls=True).over(window_fill)
    )\
  .withColumn(
    "cotistas_investidores_nao_residentes", 
    f.last("numero_cotistas_investidores_nao_residentes", ignorenulls=True).over(window_fill)
    )\
  .withColumn(
    "cotistas_outros_fundos", 
    f.last("numero_cotistas_outros_fundos", ignorenulls=True).over(window_fill)
    )\
  .withColumn(
    "cotistas_previdencia",
    f.coalesce(f.last("numero_cotistas_entidade_aberta_previdencia_complementar", True).over(window_fill), f.lit(0)) +
    f.coalesce(f.last("numero_cotistas_entidade_fechada_previdencia_complementar", True).over(window_fill), f.lit(0))
  )\
  .withColumn(
    "pct_cotistas_pf",
    f.when(f.col("total_cotistas") > 0, f.col("cotistas_pessoa_fisica") / f.col("total_cotistas"))
  )\
  .withColumn("cotistas_lag", f.lag("total_cotistas").over(window_fundo))\
  .withColumn(
    "variacao_cotistas_mes",
    f.col("total_cotistas") - f.col("cotistas_lag")
  )

### 8. Composição do Ativo (do ativo_passivo)

In [0]:
gold_cubo_fii_mensal = gold_cubo_fii_mensal.fillna({
    "total_necessidades_liquidez": 0,
    "disponibilidades": 0,
    "total_investido": 0,
    "direitos_bens_imoveis": 0,
    "imoveis_renda_acabados": 0,
    "imoveis_renda_construcao": 0,
    "cri": 0,
    "lci": 0,
    "fii": 0,
    "total_passivo": 0
})

In [0]:
gold_cubo_fii_mensal = gold_cubo_fii_mensal\
    .withColumn(
        "pct_imoveis_ativo",
        f.when(f.col("total_investido") > 0, f.col("direitos_bens_imoveis") / f.col("total_investido"))
    )\
    .withColumn(
        "pct_papel_ativo",
        f.when(f.col("total_investido") > 0, (f.col("cri") + f.col("lci") + f.col("lci_lca")) / f.col("total_investido").cast("double") ) 
    )\
    .withColumn(
        "pct_fii_ativo",
        f.when(f.col("total_investido") > 0, f.col("fii") / f.col("total_investido")) 
    )\
    .withColumn(
        "pct_liquidez",
        f.when(f.col("total_investido") > 0, f.col("total_necessidades_liquidez") / f.col("total_investido")) #
    )

### 9. Seleção de Colunas

In [0]:
gold_cubo_fii_mensal = gold_cubo_fii_mensal.select(

    # =========================
    # BLOCO 1 — IDENTIFICAÇÃO
    # =========================
    "cnpj_fundo_classe",
    "data_referencia",
    "ano_mes",

    # =========================
    # BLOCO 2 — RENTABILIDADE
    # =========================
    "rentabilidade_efetiva_mes",
    "rentabilidade_patrimonial_mes",
    "dividend_yield_mes",
    "rentabilidade_efetiva_12m",
    "dividend_yield_12m",
    "cdi_acum_mes",
    "alpha_vs_cdi_mes",
    "alpha_vs_cdi_12m",

    # =========================
    # BLOCO 3 — PATRIMÔNIO
    # =========================
    "patrimonio_liquido",
    "valor_ativo",
    "cotas_emitidas",
    "valor_patrimonial_cotas",
    "variacao_pl_mes",
    "variacao_vpc_mes",
    "taxa_administracao_pct",
    "amortizacao_cotas_mes",

    # =========================
    # BLOCO 4 — COTISTAS
    # =========================
    "total_cotistas",
    "cotistas_pessoa_fisica",
    "cotistas_pessoa_juridica",
    "cotistas_investidores_nao_residentes",
    "cotistas_previdencia",
    "cotistas_outros_fundos",
    "pct_cotistas_pf",
    "variacao_cotistas_mes",

    # =========================
    # BLOCO 5 — ATIVO/PASSIVO
    # =========================
    "total_necessidades_liquidez",
    "disponibilidades",
    "total_investido",
    "direitos_bens_imoveis",
    "imoveis_renda_acabados",
    "imoveis_renda_construcao",
    "cri",
    "lci",
    "fii",
    "total_passivo",
    "pct_imoveis_ativo",
    "pct_papel_ativo",
    "pct_fii_ativo",
    "pct_liquidez"
)

In [0]:
display(gold_cubo_fii_mensal)

### 10. Salvando os Dados

In [0]:

log.info(f"Iniciando a escrita da dimensão unificada em: {GOLD_PATH}") 

PipelineConfig.gravar_cubo_gold(
    spark=spark,
    df_cubo=gold_cubo_fii_mensal,
    tabela_destino=GOLD_PATH,
    zorder_cols=["cnpj_fundo_classe"]
)


log.info(f"Processamento da {GOLD_PATH} concluído com sucesso!")

In [0]:
%sql
select
    *
from workspace.case_spark_cvm.gold_cubo_fii_mensal